In [1]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-nep poppler-utils
!pip install -q pdf2image pytesseract tqdm

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 121026 files and directories currently installed.)
Preparing to unpack .../libpoppler-private-dev_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler-private-dev:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Preparing to unpack .../libpoppler-dev_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler-dev:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Preparing to unpack .../libpoppler118_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler118:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Selecting previously unselected package poppler-utils.
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.13) ...
Selecting previously unselected package tesseract-ocr-nep.
Preparing to unpa

In [2]:
!tesseract --list-langs

List of available languages (3):
eng
nep
osd


In [3]:
from pathlib import Path

PDF_DIR = Path("/kaggle/input/datasets/srinivasu2005/nepali-books")

pdf_files = sorted(PDF_DIR.rglob("*.pdf"))

print(f"Found {len(pdf_files)} PDF files\n")

for pdf in pdf_files:
    print(pdf)

Found 10 PDF files

/kaggle/input/datasets/srinivasu2005/nepali-books/-17 .pdf
/kaggle/input/datasets/srinivasu2005/nepali-books/2-Abhilekha 8-2075.pdf
/kaggle/input/datasets/srinivasu2005/nepali-books/Bhimsen-Thapa-ra-Tatkalin-Nepal-by-Chittaranjan-Nepali-.pdf
/kaggle/input/datasets/srinivasu2005/nepali-books/Khagaendra Final Ph.D Thesis 2016.pdf
/kaggle/input/datasets/srinivasu2005/nepali-books/MunaMadanBookLaxmiPrasadDevkota_text.pdf
/kaggle/input/datasets/srinivasu2005/nepali-books/Nepali Kathaawlokan_0.pdf
/kaggle/input/datasets/srinivasu2005/nepali-books/Nepali-Lipi-Vihas-by-Shankar-Man-Rajvamshi.pdf
/kaggle/input/datasets/srinivasu2005/nepali-books/Sadananda_PHD_Thesis.pdf
/kaggle/input/datasets/srinivasu2005/nepali-books/Siddhi-Ramayana-Kishkindha-Kanda-Siddhidas-Amatya_text.pdf
/kaggle/input/datasets/srinivasu2005/nepali-books/Tibbat-mein-Bauddha-Dharma-by-Rahul-Sankrityayana_text.pdf


In [4]:
import os
import re
import json
import time
import gc
from pathlib import Path

from tqdm import tqdm
from pdf2image import convert_from_path
import pytesseract


LANG_CODE = "nep"


def postprocess_ocr_text(text: str) -> str:
    """
    Basic cleanup based on your original script.
    """

    # Remove standalone page numbers
    text = re.sub(
        r"^\s*\d{1,4}\s*$",
        "",
        text,
        flags=re.MULTILINE
    )

    # Remove very short lines
    lines = text.split("\n")
    cleaned_lines = []

    for line in lines:
        stripped = line.strip()

        if len(stripped) > 10 or stripped == "":
            cleaned_lines.append(stripped)

    text = "\n".join(cleaned_lines)

    # Join hyphenated words across line breaks
    text = re.sub(
        r"-\s*\n\s*",
        "",
        text
    )

    # Collapse excessive spaces
    text = re.sub(r" {2,}", " ", text)

    # Collapse excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Keep lines containing Devanagari text
    devanagari_range = r"[\u0900-\u097F]"

    final_lines = []

    for line in text.split("\n"):

        stripped = line.strip()

        if not stripped:
            final_lines.append("")
            continue

        devanagari_count = len(
            re.findall(devanagari_range, stripped)
        )

        total_alpha = len(
            re.findall(r"\w", stripped)
        )

        if total_alpha > 0 and devanagari_count / total_alpha > 0.3:
            final_lines.append(stripped)

    return "\n".join(final_lines).strip()

In [5]:
def process_pdf(
    pdf_path,
    output_dir,
    batch_size=10,
    dpi=300
):
    """
    OCR one PDF and save its result as a JSON file.

    Returns metadata about the processed book.
    """

    pdf_path = Path(pdf_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_file = output_dir / f"{pdf_path.stem}.json"

    # Skip if already successfully processed
    if output_file.exists():
        print(f"Skipping already processed: {pdf_path.name}")

        with open(output_file, "r", encoding="utf-8") as f:
            return json.load(f)

    print("=" * 70)
    print(f"Processing: {pdf_path.name}")
    print("=" * 70)

    start_time = time.time()

    # --------------------------------------------------
    # Find page count without loading all pages
    # --------------------------------------------------

    from pdf2image.pdf2image import pdfinfo_from_path

    info = pdfinfo_from_path(str(pdf_path))
    total_pages = int(info["Pages"])

    print(f"Pages: {total_pages}")
    print(f"Batch size: {batch_size}")
    print()

    page_texts = []

    # --------------------------------------------------
    # Process batches
    # --------------------------------------------------

    for start_page in range(1, total_pages + 1, batch_size):

        end_page = min(
            start_page + batch_size - 1,
            total_pages
        )

        print(
            f"Pages {start_page}-{end_page} "
            f"/ {total_pages}"
        )

        batch_start = time.time()

        try:
            images = convert_from_path(
                str(pdf_path),
                dpi=dpi,
                fmt="png",
                first_page=start_page,
                last_page=end_page,
                thread_count=2
            )

        except Exception as e:
            print(
                f"ERROR rendering pages "
                f"{start_page}-{end_page}: {e}"
            )

            continue

        # ----------------------------------------------
        # OCR batch
        # ----------------------------------------------

        for image in tqdm(
            images,
            desc=f"OCR {start_page}-{end_page}",
            unit="page"
        ):

            try:

                text = pytesseract.image_to_string(
                    image,
                    lang=LANG_CODE,
                    config="--psm 1"
                )

                page_texts.append(text)

            except Exception as e:

                print(
                    f"OCR error: {e}"
                )

                page_texts.append("")

        # Free image memory
        del images
        gc.collect()

        batch_time = time.time() - batch_start

        pages_done = end_page

        pages_per_min = (
            pages_done /
            max((time.time() - start_time) / 60, 0.001)
        )

        remaining_pages = total_pages - pages_done

        eta_minutes = (
            remaining_pages / pages_per_min
            if pages_per_min > 0
            else 0
        )

        print(
            f"Speed: {pages_per_min:.2f} pages/min | "
            f"ETA: {eta_minutes:.1f} min"
        )

        print()

    # --------------------------------------------------
    # Combine OCR
    # --------------------------------------------------

    raw_text = "\n\n".join(page_texts)

    cleaned_text = postprocess_ocr_text(raw_text)

    # --------------------------------------------------
    # Save
    # --------------------------------------------------

    result = {
        "id": pdf_path.name,
        "title": pdf_path.stem,
        "text": cleaned_text,
        "source": "ocr",
        "language": "nepali",
        "pages": total_pages,
        "raw_length": len(raw_text),
        "cleaned_length": len(cleaned_text)
    }

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            result,
            f,
            ensure_ascii=False,
            indent=2
        )

    elapsed = (time.time() - start_time) / 60

    print(f"Completed: {pdf_path.name}")
    print(f"Time: {elapsed:.2f} minutes")
    print(f"Characters: {len(cleaned_text):,}")
    print(f"Saved: {output_file}")
    print()

    return result

In [6]:
PDF_DIR = Path(
    "/kaggle/input/datasets/srinivasu2005/nepali-books"
)

pdf_files = sorted(
    PDF_DIR.rglob("*.pdf")
)

print(f"Found {len(pdf_files)} PDFs\n")

for i, pdf in enumerate(pdf_files, 1):
    print(f"{i:3}. {pdf.name}")

Found 10 PDFs

  1. -17 .pdf
  2. 2-Abhilekha 8-2075.pdf
  3. Bhimsen-Thapa-ra-Tatkalin-Nepal-by-Chittaranjan-Nepali-.pdf
  4. Khagaendra Final Ph.D Thesis 2016.pdf
  5. MunaMadanBookLaxmiPrasadDevkota_text.pdf
  6. Nepali Kathaawlokan_0.pdf
  7. Nepali-Lipi-Vihas-by-Shankar-Man-Rajvamshi.pdf
  8. Sadananda_PHD_Thesis.pdf
  9. Siddhi-Ramayana-Kishkindha-Kanda-Siddhidas-Amatya_text.pdf
 10. Tibbat-mein-Bauddha-Dharma-by-Rahul-Sankrityayana_text.pdf


In [7]:
PDF_DIR = Path(
    "/kaggle/input/datasets/srinivasu2005/nepali-books"
)

pdf_files = sorted(
    PDF_DIR.rglob("*.pdf")
)

print(f"Found {len(pdf_files)} PDFs\n")

for i, pdf in enumerate(pdf_files, 1):
    print(f"{i:3}. {pdf.name}")

Found 10 PDFs

  1. -17 .pdf
  2. 2-Abhilekha 8-2075.pdf
  3. Bhimsen-Thapa-ra-Tatkalin-Nepal-by-Chittaranjan-Nepali-.pdf
  4. Khagaendra Final Ph.D Thesis 2016.pdf
  5. MunaMadanBookLaxmiPrasadDevkota_text.pdf
  6. Nepali Kathaawlokan_0.pdf
  7. Nepali-Lipi-Vihas-by-Shankar-Man-Rajvamshi.pdf
  8. Sadananda_PHD_Thesis.pdf
  9. Siddhi-Ramayana-Kishkindha-Kanda-Siddhidas-Amatya_text.pdf
 10. Tibbat-mein-Bauddha-Dharma-by-Rahul-Sankrityayana_text.pdf


In [8]:
OUTPUT_DIR = Path(
    "/kaggle/working/nepali_ocr"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Output directory:")
print(OUTPUT_DIR)

Output directory:
/kaggle/working/nepali_ocr


In [9]:
test_pdf = pdf_files[0]

result = process_pdf(
    test_pdf,
    OUTPUT_DIR,
    batch_size=10,
    dpi=300
)

Processing: -17 .pdf
Pages: 159
Batch size: 10

Pages 1-10 / 159


OCR 1-10: 100%|██████████| 10/10 [00:26<00:00,  2.69s/page]


Speed: 16.76 pages/min | ETA: 8.9 min

Pages 11-20 / 159


OCR 11-20: 100%|██████████| 10/10 [00:41<00:00,  4.17s/page]


Speed: 13.46 pages/min | ETA: 10.3 min

Pages 21-30 / 159


...

Speed: 11.59 pages/min | ETA: 1.6 min

Pages 141-150 / 159


OCR 141-150: 100%|██████████| 10/10 [00:31<00:00,  3.11s/page]


Speed: 11.78 pages/min | ETA: 0.8 min

Pages 151-159 / 159


OCR 151-159: 100%|██████████| 9/9 [00:39<00:00,  4.43s/page]

Speed: 11.73 pages/min | ETA: 0.0 min

Completed: -17 .pdf
Time: 13.56 minutes
Characters: 185,783
Saved: /kaggle/working/nepali_ocr/-17 .json



In [10]:
print("Title:", result["title"])
print("Pages:", result["pages"])
print("Raw characters:", result["raw_length"])
print("Clean characters:", result["cleaned_length"])

print("\n" + "=" * 70)
print("OCR SAMPLE")
print("=" * 70)

print(result["text"][:5000])

Title: -17 
Pages: 159
Raw characters: 194114
Clean characters: 185783

OCR SAMPLE
अभिलेख विश्वसम्पदा हो

यसको जगेनी गर्नु हाम्रो कर्तब्य हो
अधिलेखकीो पुरक्षा गरौं
संस्कृतिको रक्षा गरौं ।

- बाच््रिस उष्ति रेक्स

वर्ष १७ ) २०५६ पूर्णाङ्क १७

स्सरूप्याल्ट्ष्छ-क्बण्ज्टट

प्रचान सम्पादक सम्पादक
तुथ्ी सानुनानी कसगकार विष्णुकान्त शर्मा
जगनभ्नाय उपाध्याय

प्सखकाकाष्क
श्री ५ को सरकार

थवा, खेलकृव तया शंस्कृति मन्त्रालय
पुरातत्व विद्या

एाख्थ्ट्रि आस्निसेष्बार रस

प्रकाशन मितिः २०१६ साल ध्राश्विण १७ गते

प्रकाशित प्रतिः- ५००

मूल्य रू, :- २४।मुद्दक :- पसंख्हस्टा प्रेस्त,
ख १-०३ टेबहात, काठमाडौं -॥
हार २३०४४७९, २२८२०८ (पसागत )

२०२० सालमा स्थापना भएको राष्ट्रिय प्रभिलेखालप धाजदेखि ३२ आौं वर्षमा
पदार्पण गरिरहेछ । बढी उपलक्ष्यमा राष्ट्रिय प्रभिलेखालपको संग्रहमा प्रएका
प्रशासनिक, ऐतिहासिक, सांस्कृतिक प्रादि सामग्रीहको जानकारौ दिने मूल
भ्रभिप्राय बोको “भ्रभिलेख' को सत्रौं प्रकु पाठक वगं सामु राछ्न पाएको छु । यस
प्रकुमा पनि लालमोहर, ताडपत्र तपसुक, ऐतिहासिक र ग्यनुहन्धानमूलक रचनाहरू
समावेश गरिएको छ ।

In [11]:
all_results = []

overall_start = time.time()

for i, pdf_path in enumerate(pdf_files, 1):

    print("\n")
    print("#" * 80)
    print(f"BOOK {i}/{len(pdf_files)}")
    print("#" * 80)

    try:

        result = process_pdf(
            pdf_path,
            OUTPUT_DIR,
            batch_size=10,
            dpi=300
        )

        all_results.append(result)

    except Exception as e:

        print(
            f"FAILED: {pdf_path.name}"
        )

        print(
            f"Error: {e}"
        )

        continue

overall_minutes = (
    time.time() - overall_start
) / 60

print("\n")
print("=" * 80)
print("ALL OCR COMPLETE")
print("=" * 80)

print(
    f"Successfully processed: "
    f"{len(all_results)}/{len(pdf_files)}"
)

print(
    f"Total time: "
    f"{overall_minutes:.2f} minutes"
)



################################################################################
BOOK 1/10
################################################################################
Skipping already processed: -17 .pdf


################################################################################
BOOK 2/10
################################################################################
Processing: 2-Abhilekha 8-2075.pdf
Pages: 162
Batch size: 10

Pages 1-10 / 162


OCR 1-10: 100%|██████████| 10/10 [00:36<00:00,  3.61s/page]


Speed: 11.10 pages/min | ETA: 13.7 min

Pages 11-20 / 162


OCR 11-20: 100%|██████████| 10/10 [01:11<00:00,  7.18s/page]


Speed: 8.13 pages/min | ETA: 17.5 min

Pages 21-30 / 162


...

Speed: 8.81 pages/min | ETA: 1.4 min

Pages 71-80 / 82


OCR 71-80: 100%|██████████| 10/10 [00:27<00:00,  2.73s/page]


Speed: 9.19 pages/min | ETA: 0.2 min

Pages 81-82 / 82


OCR 81-82: 100%|██████████| 2/2 [00:08<00:00,  4.06s/page]

Speed: 9.00 pages/min | ETA: 0.0 min

Completed: Tibbat-mein-Bauddha-Dharma-by-Rahul-Sankrityayana_text.pdf
Time: 9.11 minutes
Characters: 105,948
Saved: /kaggle/working/nepali_ocr/Tibbat-mein-Bauddha-Dharma-by-Rahul-Sankrityayana_text.json



ALL OCR COMPLETE
Successfully processed: 10/10
Total time: 260.00 minutes


In [12]:
JSONL_PATH = Path(
    "/kaggle/working/nepali_ocr/extracted.jsonl"
)

json_files = sorted(
    OUTPUT_DIR.glob("*.json")
)

count = 0

with open(
    JSONL_PATH,
    "w",
    encoding="utf-8"
) as out:

    for json_file in json_files:

        # Don't accidentally read an old combined JSONL
        if json_file.name == "extracted.json":
            continue

        try:

            with open(
                json_file,
                "r",
                encoding="utf-8"
            ) as f:

                data = json.load(f)

            out.write(
                json.dumps(
                    data,
                    ensure_ascii=False
                ) + "\n"
            )

            count += 1

        except Exception as e:

            print(
                f"Could not combine {json_file.name}: {e}"
            )

print(f"Combined {count} documents")
print(f"Output: {JSONL_PATH}")

Combined 10 documents
Output: /kaggle/working/nepali_ocr/extracted.jsonl


In [13]:
total_chars = 0
total_pages = 0

with open(
    JSONL_PATH,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        item = json.loads(line)

        total_chars += len(
            item["text"]
        )

        total_pages += item["pages"]

print(f"Documents : {count:,}")
print(f"Pages     : {total_pages:,}")
print(f"Characters: {total_chars:,}")

Documents : 10
Pages     : 1,626
Characters: 2,514,045


In [14]:
print(
    f"Approx chars: "
    f"{total_chars:,}"
)

print(
    f"Rough 4-char estimate: "
    f"{total_chars / 4:,.0f} tokens"
)

Approx chars: 2,514,045
Rough 4-char estimate: 628,511 tokens


In [15]:
print("Completed books:\n")

for json_file in sorted(OUTPUT_DIR.glob("*.json")):

    if json_file.name == "extracted.json":
        continue

    with open(
        json_file,
        "r",
        encoding="utf-8"
    ) as f:

        item = json.load(f)

    print(
        f"{item['title']}: "
        f"{item['pages']} pages, "
        f"{item['cleaned_length']:,} chars"
    )

Completed books:

-17 : 159 pages, 185,783 chars
2-Abhilekha 8-2075: 162 pages, 244,564 chars
Bhimsen-Thapa-ra-Tatkalin-Nepal-by-Chittaranjan-Nepali-: 142 pages, 416,617 chars
Khagaendra Final Ph.D Thesis 2016: 455 pages, 928,387 chars
MunaMadanBookLaxmiPrasadDevkota_text: 51 pages, 46,524 chars
Nepali Kathaawlokan_0: 121 pages, 387,838 chars
Nepali-Lipi-Vihas-by-Shankar-Man-Rajvamshi: 100 pages, 97,998 chars
Sadananda_PHD_Thesis: 291 pages, 41,298 chars
Siddhi-Ramayana-Kishkindha-Kanda-Siddhidas-Amatya_text: 63 pages, 59,088 chars
Tibbat-mein-Bauddha-Dharma-by-Rahul-Sankrityayana_text: 82 pages, 105,948 chars


In [16]:
from IPython.display import FileLink

FileLink(
    str(JSONL_PATH)
)

/kaggle/working/nepali_ocr/extracted.jsonl